# sin(x) EML Fourier Floor Asymptotics — Session 36

**Question:** As N (max internal nodes in EML dictionary) grows from 1→6, does the floor MSE for sin(x) approach 0 (DENSE) or plateau (SEPARATION)?

The Pumping Lemma predicts a floor because EML atoms have period π, not 2π.
But the lemma bounds *single trees* — not linear combinations. This notebook answers whether the linear span can escape the barrier.

In [ ]:
import sys, math
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from monogate.frontiers.eml_fourier import build_eml_dictionary, _generate_shapes
from monogate.frontiers.eml_fourier_v2 import build_eml_dictionary_v2

# Dictionary sizes by N
print(f'{'N':>4}  {'Raw':>8}  {'V1 valid':>10}  {'V2 valid':>10}')
for n in range(1, 7):
    shapes = _generate_shapes(n)
    raw = len(shapes) * (1 << (n + 1))
    v1 = build_eml_dictionary(n)
    v2 = build_eml_dictionary_v2(n)
    print(f'{n:>4}  {raw:>8}  {len(v1):>10}  {len(v2):>10}')

In [ ]:
from monogate.frontiers.eml_fourier_v2 import eml_fourier_search_v2
import time

floor_results = {}
for n in [1, 2, 3, 4, 5]:
    t0 = time.time()
    r = eml_fourier_search_v2(math.sin, 'sin', max_internal_nodes=n, max_K=30, detect_floor=True)
    elapsed = time.time() - t0
    floor_mse = r.floor_mse if math.isfinite(r.floor_mse) else r.mse_test
    floor_results[n] = {'floor_mse': floor_mse, 'floor_K': r.floor_K,
                        'n_indep': r.n_independent_atoms, 'elapsed': elapsed}
    print(f'N={n}: floor_mse={floor_mse:.4e}  floor_K={r.floor_K}  '
          f'indep={r.n_independent_atoms}  t={elapsed:.1f}s')

In [ ]:
# Floor decay plot
ns = sorted(floor_results)
mses = [floor_results[n]['floor_mse'] for n in ns]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(ns, mses, 'o-', color='steelblue', linewidth=2, markersize=8)

if len(ns) >= 3:
    coeffs = np.polyfit(ns, np.log(mses), 1)
    trend = np.exp(np.polyval(coeffs, ns))
    ax.semilogy(ns, trend, '--', color='red', alpha=0.7,
                label=f'exp fit: MSE ~ exp({coeffs[0]:.2f}·N)')
    ax.legend(fontsize=11)

for n, m in zip(ns, mses):
    ax.annotate(f'{m:.1e}', (n, m), textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('N (max internal nodes)', fontsize=12)
ax.set_ylabel('Floor MSE', fontsize=12)
ax.set_title('sin(x) EML Fourier Floor vs Dictionary Depth', fontsize=13)
ax.set_xticks(ns)
ax.grid(True, alpha=0.4, which='both')
plt.tight_layout()
plt.savefig('../results/sin_floor_asymptotics.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Residual waterfall: sin(x) residual at N=2,3,4
from monogate.frontiers.eml_fourier import _eval_tree

xs = np.linspace(0.05, 2.0 * math.pi, 400)
y_true = np.sin(xs)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
colors = ['#e74c3c', '#e67e22', '#2ecc71']

for ax, n, color in zip(axes, [2, 3, 4], colors):
    r = eml_fourier_search_v2(math.sin, 'sin', max_internal_nodes=n, max_K=30, detect_floor=True)
    y_pred = np.zeros_like(xs)
    for c, a in zip(r.coefficients, r.atoms):
        for i, x in enumerate(xs):
            v = _eval_tree(a.ops, a.leaf_mask, float(x))
            y_pred[i] += c * (v if v is not None and math.isfinite(v) else 0.0)
    residual = y_true - y_pred
    ax.plot(xs, residual, color=color, linewidth=1.2)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('Residual', fontsize=10)
    floor_mse = r.floor_mse if math.isfinite(r.floor_mse) else r.mse_test
    ax.set_title(f'N<={n}, K={r.K}, floor_MSE={floor_mse:.3e}', fontsize=11)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('x', fontsize=11)
fig.suptitle('sin(x) EML Fourier Residuals by Depth', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary

| N | Floor MSE | Floor K | Independent atoms |
|---|-----------|---------|-------------------|
| 1 | ~2e-2     | 1       | 3                 |
| 2 | ~4e-5     | 8       | 14                |
| 3 | ~1.65e-6  | 11      | ~60               |
| 4 | TBD       | TBD     | ~280              |
| 5 | TBD       | TBD     | ~1400             |

**VERDICT:** See `sin_floor_asymptotics.py` for the DENSE / SEPARATION determination.

The residual at every N has dominant period ≈ π (Pumping Lemma signature). The question is
whether adding deeper atoms (which can represent half-π oscillations) eventually closes the gap.